# W03: 메뉴 설명 & SNS 카피 자동화

메뉴 정보를 입력해 배달앱용 설명(80자), 인스타 캡션을 생성하고 결과를 ZIP으로 묶어 저장합니다.


In [ ]:
import io
import json
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm

try:
    import google.generativeai as genai
except Exception:
    genai = None

GEMINI_MODEL = "gemini-2.0-flash"

for fp in fm.findSystemFonts():
    try:
        fm.fontManager.addfont(fp)
    except Exception:
        continue


def safe_upload():
    try:
        from google.colab import files
    except Exception:
        print("Colab 환경이 아니라 files.upload를 사용할 수 없습니다.")
        return {}
    try:
        return files.upload()
    except Exception as e:
        print(f"files.upload() 취소/실패: {e}")
        print("files.upload() 취소되어 기본 데이터 사용")
        return {}


def use_gemini(prompt: str, fallback: str) -> str:
    if not genai:
        return fallback
    key = os.getenv("GOOGLE_API_KEY", "").strip()
    if not key:
        return fallback
    try:
        genai.configure(api_key=key)
        model = genai.GenerativeModel(GEMINI_MODEL)
        result = model.generate_content(prompt)
        return getattr(result, "text", "").strip() or fallback
    except Exception as e:
        print(f"Gemini 호출 실패: {e}")
        return fallback

menus = [
    {"name": "치킨버거", "price": 14000, "ingredient": "닭다리살, 빵, 양상추"},
    {"name": "토마토파스타", "price": 13000, "ingredient": "토마토소스, 파스타, 치즈"},
    {"name": "초코케이크", "price": 9000, "ingredient": "초코, 크림, 버터"},
]

rows = []
for m in menus:
    prompt_desc = f"메뉴:{m['name']}, 가격:{m['price']}, 주재료:{m['ingredient']}. 배달앱용 설명 80자 이내"
    prompt_cap = f"메뉴:{m['name']} 인스타그램 캡션(이모지, 해시태그 포함) 1문단"
    desc = use_gemini(prompt_desc, f"[{m['name']}] "+f"{m['ingredient']}로 만든 메뉴, {m['price']}원 간편 배달 특가 제공")
    cap = use_gemini(prompt_cap, f"[{m['name']}] 오늘의 추천! #맛집 #배달" )
    rows.append({"메뉴": m["name"], "가격": m["price"], "배달앱설명": desc, "인스타캡션": cap})

out = pd.DataFrame(rows)
Path("menu_copy_outputs").mkdir(exist_ok=True)
for i, r in out.iterrows():
    txt = f"메뉴명: {r['메뉴']}
가격: {r['가격']}
설명: {r['배달앱설명']}
캡션: {r['인스타캡션']}
"
    Path("menu_copy_outputs").joinpath(f"{r['메뉴']}.txt").write_text(txt, encoding="utf-8")

with __import__('zipfile').ZipFile("menu_copy_outputs.zip", "w") as z:
    for file in Path("menu_copy_outputs").glob("*.txt"):
        z.write(file, arcname=file.name)

print(out)
print("menu_copy_outputs.zip 생성")
